# Lesson 3: any single-qubit gate on a 1D cluster wire

Each measured qubit applies one $J(\theta) = H P(-\theta)$. Four of them in a row give
$$J(\theta_3)J(\theta_2)J(\theta_1)J(0) = HP_3\,HP_2\,HP_1\,H = (HP_3H)\,P_2\,(HP_1H) \propto R_x(-\theta_3)\,R_z(-\theta_2)\,R_x(-\theta_1),$$
using $HP(\phi)H \propto R_x(\phi)$ and $P(\phi)\propto R_z(\phi)$. Every single-qubit unitary has an Euler decomposition
$U \propto R_x(c)R_z(b)R_x(a)$, so a **5-qubit wire** (input + 3 rotation qubits + 1 extra Hadamard step + output) does any
single-qubit gate. This is the construction of [Raussendorf, Browne & Briegel, arXiv:quant-ph/0301052](https://arxiv.org/abs/quant-ph/0301052).

The code is in `run_wire` and `rotation_angles` in `mbqc/patterns.py`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mbqc import *

rng = np.random.default_rng(2026)

In [2]:
from scipy.stats import unitary_group  # only used to draw a random U

U = unitary_group.rvs(2, random_state=7)
a, b, c = euler_xzx(U)
print("Euler angles (a, b, c):", np.round([a, b, c], 4))
print("check Rx(c) Rz(b) Rx(a) = U up to phase:", round(fidelity((Rx(c) @ Rz(b) @ Rx(a))[:, 0], U[:, 0]), 12))

angles = rotation_angles(U)
print("measurement angles along the wire:", np.round(angles, 4))

Euler angles (a, b, c): [ 3.1004  0.9017 -3.1301]
check Rx(c) Rz(b) Rx(a) = U up to phase: 1.0
measurement angles along the wire: [ 0.     -3.1004 -0.9017  3.1301]


## Check every one of the 16 outcome branches

With the Pauli frame adapting the signs, all $2^4$ branches give $U|\psi\rangle$ exactly. Without adaptation most branches
are wrong.

In [3]:
import itertools

psi = random_state(1, rng)
branches = list(itertools.product((0, 1), repeat=4))
good = [fidelity(run_wire(psi, angles, outcomes=o)[0], U @ psi) for o in branches]
bad = [fidelity(run_wire(psi, angles, outcomes=o, adaptive=False)[0], U @ psi) for o in branches]
print("adaptive:     min fidelity over branches =", round(min(good), 12))
print("non-adaptive: fidelities =", np.round(bad, 3))

adaptive:     min fidelity over branches = 1.0
non-adaptive: fidelities = [1.    1.    1.    1.    0.466 0.466 0.473 0.473 1.    1.    1.    1.
 0.456 0.456 0.45  0.45 ]


## Which measurements actually have to wait?

Only angles that are not multiples of $\pi/2$. Flipping the sign of $0$ or $\pi$ does nothing, and flipping $\pm\pi/2$
just swaps the two outcome labels. So $X$ and $Y$ measurements (the Clifford part) can all be done at once, at the start.
This is one of the surprises of MBQC and we come back to it in lesson 5.

Let's watch a single run with random outcomes and print the frame as it evolves.

In [4]:
frame = PauliFrame()
state = QubitState.product({0: psi})
for k in range(1, 5):
    state.add(k)
for k in range(4):
    state.cz(k, k + 1)

for k, theta in enumerate(angles):
    used = frame.angle(k, theta)
    s = state.measure_xy(k, used, rng=rng)
    frame.teleport(k, k + 1, s)
    print(f"measure qubit {k} at {used:+.3f} (wanted {theta:+.3f}) -> s={s};  byproduct on qubit {k+1}: "
          f"X^{frame.x[k+1]} Z^{frame.z[k+1]}")
frame.undo(state, 4)
print("fidelity with U|psi>:", round(fidelity(state.vector(), U @ psi), 12))

measure qubit 0 at +0.000 (wanted +0.000) -> s=1;  byproduct on qubit 1: X^1 Z^0
measure qubit 1 at +3.100 (wanted -3.100) -> s=0;  byproduct on qubit 2: X^0 Z^1
measure qubit 2 at -0.902 (wanted -0.902) -> s=0;  byproduct on qubit 3: X^1 Z^0
measure qubit 3 at -3.130 (wanted +3.130) -> s=1;  byproduct on qubit 4: X^1 Z^1
fidelity with U|psi>: 1.0


## The same adaptive wire in PennyLane

`pennylane.ftqc.cond_measure` picks the measurement basis from an earlier outcome, which is exactly an adaptive
measurement. Three qubits, two measurements, with the final corrections done by `qml.cond`.

In [5]:
import pennylane as qml
from pennylane.ftqc import cond_measure, diagonalize_mcms, measure_arbitrary_basis

dev = qml.device("default.qubit")
t1, t2 = 0.4, 1.3


@diagonalize_mcms
@qml.qnode(dev, mcm_method="tree-traversal")
def wire3():
    qml.StatePrep(psi, wires=0)
    qml.H(1), qml.H(2)
    qml.CZ([0, 1]), qml.CZ([1, 2])
    m0 = measure_arbitrary_basis(0, angle=t1, plane="XY")
    m1 = cond_measure(
        m0,
        lambda: measure_arbitrary_basis(1, angle=-t2, plane="XY"),  # byproduct X present: flip the sign
        lambda: measure_arbitrary_basis(1, angle=t2, plane="XY"),
    )()
    qml.cond(m1, qml.X)(2)  # frame after two steps: X^{s1} Z^{s0}
    qml.cond(m0, qml.Z)(2)
    return [qml.expval(op(2)) for op in (qml.X, qml.Y, qml.Z)]


def bloch(v):
    return np.real([np.vdot(v, P @ v) for P in (X, Y, Z)])

print("PennyLane:", np.round(wire3(), 4))
print("NumPy    :", np.round(bloch(J(t2) @ J(t1) @ psi), 4))

PennyLane: [-0.9202  0.3866 -0.0611]
NumPy    : [-0.9202  0.3866 -0.0611]


**Takeaways**
* A 1D cluster is a wire: each measurement moves the qubit one site and applies $J(\theta)$.
* Four measurements give an arbitrary rotation via the $XZX$ Euler decomposition.
* Signs of later angles depend on earlier outcomes. A 1D cluster alone is *not* universal: it only ever holds one
  logical qubit. For that we need two dimensions.